In [1]:
def add_something(a, b):
    print(f"Adding {a} and {b} together...")
    return a + b


add_something(**{"a": 7, "b": 12})

Adding 7 and 12 together...


19

Did you know? In Python, `**` in a function call means **“take this mapping (usually a dict) and expand it into keyword arguments.”**

### What’s happening in your call

```python
add_something(**{"a": 7, "b": 12})
```

Python sees a dict with keys `"a"` and `"b"`, and expands it as if you had written:

```python
add_something(a=7, b=12)
```

So it’s not passing the dict as a single argument. It’s **unpacking** it into named parameters.

### Why it works

Because your function is defined with parameter names `a` and `b`:

```python
def add_something(a, b):
```

…and the dict has matching keys:

```python
{"a": 7, "b": 12}
```

So `a` gets `7`, `b` gets `12`.

### Rules (the important bits)

1. **Keys must be strings** (valid keyword names, effectively):

   ```python
   add_something(**{1: 7, 2: 12})  # TypeError
   ```

2. **Keys must match parameter names**, unless the function accepts arbitrary keywords:

   ```python
   add_something(**{"x": 7, "b": 12})
   # TypeError: got an unexpected keyword argument 'x'
   ```

3. You can mix normal args + `**` (but don’t give the same value twice):

   ```python
   add_something(7, **{"b": 12})      # ok -> a=7, b=12
   add_something(a=7, **{"a": 9})     # TypeError: multiple values for 'a'
   ```

4. You can also merge multiple `**` expansions:

   ```python
   add_something(**{"a": 7}, **{"b": 12})
   ```

### Related: `*` vs `**`

* `*` unpacks a sequence into **positional arguments**:

  ```python
  add_something(*[7, 12])   # same as add_something(7, 12)
  ```

* `**` unpacks a mapping into **keyword arguments**:

  ```python
  add_something(**{"a": 7, "b": 12})  # same as add_something(a=7, b=12)
  ```

If you want, I can show how this interacts with `def f(*args, **kwargs)` which is where `**` really becomes obvious.


In [2]:
import os, json
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [3]:
import truststore
truststore.inject_into_ssl()

#This is optional. I use VPN in my computer. Why I have to use this



In [4]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

MODEL  = "gpt-5-nano"  

OpenAI client ready.


In [5]:
# The model has NO access to live data — what does it do?

response = client.responses.create(
    model=MODEL,
    input="What is the weather in bengaluru right now? Give me the exact temperature.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
print(response.output_text)

I can’t fetch real-time data. Please check a reliable weather service (e.g., weather.com, AccuWeather, or a live weather app) for the current temperature in Bengaluru. If you share a link or screenshot, I can help interpret it.


In [6]:
# The model PREDICTS math — it doesn't COMPUTE it

response = client.responses.create(
    model=MODEL,
    input="What is 1247 * 83 + 19 / 3.7? Give me answer in one line (max 10 words), the exact number, upto 2 decimal places.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
print("Model says:", response.output_text)

# What Python actually computes
import math
actual = 1247 * 83 + 19 / 3.7
print(f"Actual answer: {actual}")


Model says: 103,736.49
Actual answer: 103506.13513513513


In [ ]:
button

In [7]:
# Define a simple "add" tool — we'll explain the structure in detail next
add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

# how to tell the model what function exists, and what argument it takes. 

# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)



# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: add
  Arguments: {"a":7,"b":12}
  Call ID: call_RbzQpNY2cznGFEfKa2WDuyh0
 Response:


In [8]:
def add(a, b):
    print(f"Adding {a} and {b} together...")
    return a + b

    
add(**{"a": 7, "b": 12})

Adding 7 and 12 together...


19

In [9]:
# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="Why is Earth round?",
    tools=[add_tool],
)


# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: message
 Response: Earth isn’t a perfect ball, but it is round in a general sense.
Here’s why:  - Gravity pulls matter toward the center. Over long timescales,
gravity tries to make a large enough body into a shape where the surface is at a
constant gravitational potential—i.e., as even as possible.  - Earth’s rotation
adds a twist. The spinning motion creates centrifugal force that is strongest at
the equator, causing the planet to bulge a bit. This makes Earth an oblate
spheroid: wider at the equator than from pole to pole.  - The result is a
roughly spherical shape with a small flattening. The equatorial radius is about
6378 km, the polar radius about 6357 km, giving a flattening of roughly 1/298.
The difference is about 21 km between equator and pole.  - Topography adds local
variation. Mountains, trenches, and surface irregularities shift the shape by
only tens of kilometers—tiny compar

In [20]:
# Adding more tools doesn't change the model's behavior if the question doesn't require them

add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract two numbers and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide two numbers and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b

# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the tools for any and every kind of math. Never compute math yourself.",
    #input="What is 1247 * 83 + 19 / 3.7?",
    input = "I have right now fifteen oranges, 3.14 apple, 12.86 mangoes, but out of these 2.19 apples and 5.22 mangoes got rotten. How many fresh fruits do I have now?",
    tools=[add_tool, sub_tool, mul_tool, div_tool],
    reasoning={"effort": "high"}
)


# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)


Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: subtract
  Arguments: {"a":3.14,"b":2.19}
  Call ID: call_AyWpbL5waqKdNaL8pFPzIyOf
  Type: function_call
  Function name: subtract
  Arguments: {"a":12.86,"b":5.22}
  Call ID: call_hW3yzQsXBqdIZuuzF7YFZcO9
 Response:


In [ ]:
DISPATCH = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
} # b/w schema and actual code, we need to maintain this mapping. We can automate this in future.

In [16]:
for i, item in enumerate(response.output):
    if item.type == "function_call":
        func_name = item.name
        args = item.arguments
        call_id = item.call_id
        
        if func_name in DISPATCH:
            func = DISPATCH[func_name]
            mod_args = json.loads(args)  # Convert JSON string to Python dict
            result = func(**mod_args)  # Call the function with unpacked arguments
            print(f"Result of {func_name} with arguments {args}: {result}")
        else:
            print(f"Unknown function: {func_name}")

Result of multiply with arguments {"a":1247,"b":83}: 103501
Result of divide with arguments {"a":19,"b":3.7}: 5.135135135135135


In [21]:
# Define a get_weather tool
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. 'Tel Aviv', 'London'"
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}


# Same question as before — but now the model has a tool!
response = client.responses.create(
    model=MODEL,
    input="What's the weather in Paris right now?",
    tools=[weather_tool],
)

# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: get_weather
  Arguments: {"location":"Paris"}
  Call ID: call_1RfTC2op82iZ8WD31HBZXdVZ
 Response:


In [22]:
# Our fake weather function (in production, this would call a real API)
def get_weather(location):
    fake_data = {"Tel Aviv": "28°C, sunny", "Paris": "18°C, cloudy", "London": "14°C, rain"}
    return fake_data.get(location, f"No data for {location}")

# Step 1: Ask with the weather tool
response = client.responses.create(
    model=MODEL,
    input="What's the weather like in Tel Aviv and London?",
    tools=[weather_tool],
)

# Step 2 & 3: Find all function calls, execute each
print("Model requested these calls:")
new_input = list(response.output)

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"  → {item.name}({args}) = {result}")
        
        # Step 4: Append our result
        new_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        })

# Step 5: Model composes final answer with real data
final = client.responses.create(
    model=MODEL,
    input=new_input,
    tools=[weather_tool],
)
print(f"\nFinal answer:\n{final.output_text}")

Model requested these calls:
  → get_weather({'location': 'Tel Aviv'}) = 28°C, sunny
  → get_weather({'location': 'London'}) = 14°C, rain

Final answer:
Here are the current conditions I found:

- Tel Aviv: 28°C, sunny
- London: 14°C, rain

Would you like these converted to Fahrenheit, or want a short forecast for either city?


In [24]:
# Adding more tools doesn't change the model's behavior if the question doesn't require them

add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract two numbers and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide two numbers and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b


DISPATCH = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

TOOLS = [add_tool, sub_tool, mul_tool, div_tool]





DEV_POLICY = "Use the tools for any math. Never compute math yourself."

def answer_with_math_tools_verbose(user_question: str) -> str:
    print("══════════════════════════════════════════════════════")
    print("START: User question")
    print("  ", user_question)
    print("══════════════════════════════════════════════════════")

    # Round 1: seed conversation with developer policy + user question
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "developer", "content": DEV_POLICY},
            {"role": "user", "content": user_question},
        ],
        tools=TOOLS
    )

    round_num = 1

    while True:
        print(f"\n══════════════════════════════════════════════════════")
        print(f"ROUND {round_num}: Model response items")
        print("══════════════════════════════════════════════════════")

        # Show every output item type
        for i, item in enumerate(resp.output):
            print(f"[{i}] type = {item.type}")

            if item.type == "function_call":
                print(f"    name     = {item.name}")
                print(f"    call_id  = {item.call_id}")
                print(f"    arguments(raw JSON string) = {item.arguments}")

            elif item.type == "message":
                # Some SDKs represent assistant text as message items
                # output_text is the easiest way to get the final combined text.
                try:
                    print(f"    content = {item.content}")
                except Exception:
                    pass

        calls = [item for item in resp.output if item.type == "function_call"]

        # If no tool calls, we’re done; return final user-facing text
        if not calls:
            print("\n✅ No function calls. Final assistant text:")
            print(resp.output_text)
            return resp.output_text

        print("\n→ Model requested tool calls:")
        for call in calls:
            print(f"  - {call.name}({call.arguments})  [call_id={call.call_id}]")

        # Execute all calls and prepare tool outputs
        tool_outputs = []
        print("\n→ Executing tools locally (your server/app):")
        for call in calls:
            fn = DISPATCH.get(call.name)
            args = json.loads(call.arguments)

            try:
                result = fn(**args)
                payload = {"ok": True, "result": result}
                print(f"  ✓ {call.name}(**{args}) -> {result}")
            except Exception as e:
                payload = {"ok": False, "error": str(e)}
                print(f"  ✗ {call.name}(**{args}) -> ERROR: {e}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                # output is a string; keep it JSON for readability + structure
                "output": json.dumps(payload),
            })

        print("\n→ Sending tool outputs back to the model:")
        for out in tool_outputs:
            pretty_print(out)

        # Continue conversation, ONLY sending tool outputs (chained with previous_response_id)
        resp = client.responses.create(
            model=MODEL,
            previous_response_id=resp.id,
            input=tool_outputs,
            tools=TOOLS,
            reasoning={"effort": "minimal"},
        )

        round_num += 1

# Example
#final_text = answer_with_math_tools_verbose("What is 1247 * 83 + 19 / 3.7?")
final_text = answer_with_math_tools_verbose("I have 15 oranges, 10 apples, 3 bananas and 4 avocados.  How many fruits do I have now?")

══════════════════════════════════════════════════════
START: User question
   I have 15 oranges, 10 apples, 3 bananas and 4 avocados.  How many fruits do I have now?
══════════════════════════════════════════════════════

══════════════════════════════════════════════════════
ROUND 1: Model response items
══════════════════════════════════════════════════════
[0] type = reasoning
[1] type = function_call
    name     = add
    call_id  = call_bld6fOn56Ud2eyHSjwRlpeiM
    arguments(raw JSON string) = {"a":15,"b":10}
[2] type = function_call
    name     = add
    call_id  = call_RTUca17CX2l5eXY1yK7toomR
    arguments(raw JSON string) = {"a":3,"b":4}

→ Model requested tool calls:
  - add({"a":15,"b":10})  [call_id=call_bld6fOn56Ud2eyHSjwRlpeiM]
  - add({"a":3,"b":4})  [call_id=call_RTUca17CX2l5eXY1yK7toomR]

→ Executing tools locally (your server/app):
  ✓ add(**{'a': 15, 'b': 10}) -> 25
  ✓ add(**{'a': 3, 'b': 4}) -> 7

→ Sending tool outputs back to the model:
{'type': 'function_call